# Clase 111 — Gradient clipping

Limitar la **norma** o el **valor** de los gradientes antes de actualizar pesos, como protección contra exploding gradients (crítico en RNN/LSTM y LLMs). `clipnorm` preserva la dirección; `clipvalue` recorta por elemento.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`.

## 1. Configurar clipping en el optimizer

Cualquier optimizer Keras acepta `clipnorm`, `clipvalue` o `global_clipnorm`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

opt_norm   = keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
opt_value  = keras.optimizers.Adam(learning_rate=1e-3, clipvalue=0.5)
opt_global = keras.optimizers.Adam(learning_rate=1e-3, global_clipnorm=1.0)
print("clipnorm:", opt_norm.clipnorm,
      "| clipvalue:", opt_value.clipvalue,
      "| global_clipnorm:", opt_global.global_clipnorm)

## 2. `clipnorm` preserva la dirección

Si `||g|| > c`, se reescala `g ← g·c/||g||`: la dirección del vector no cambia.

In [ ]:
g = tf.constant([3.0, 4.0])                       # ||g|| = 5
recortado, norma = tf.clip_by_global_norm([g], clip_norm=1.0)
print("norma original:", float(norma), "→ tras clip:", float(tf.norm(recortado[0])))
print("dirección tras clip :", (recortado[0] / tf.norm(recortado[0])).numpy().round(3))
print("dirección original  :", (g / tf.norm(g)).numpy().round(3), "(idéntica)")

## 3. `clipvalue` recorta por elemento y cambia la dirección

Cada componente se limita a `[-c, +c]`, lo que puede alterar la dirección del gradiente.

In [ ]:
g = tf.constant([3.0, 0.1])
por_valor = tf.clip_by_value(g, -0.5, 0.5)
print("clipvalue por elemento:", por_valor.numpy())
print("dirección original :", (g / tf.norm(g)).numpy().round(3))
print("dirección recortada:", (por_valor / tf.norm(por_valor)).numpy().round(3),
      "(cambió)")

## 4. Forzar exploding y recortarlo

Con `Adam(lr=10.0)` el modelo tiende a `loss=nan`; agregar `clipnorm=1.0` acota la explosión (aunque el LR sigue mal calibrado).

In [ ]:
def mlp(optimizer):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(300, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(100, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

sin_clip = mlp(keras.optimizers.Adam(learning_rate=10.0))
con_clip = mlp(keras.optimizers.Adam(learning_rate=10.0, clipnorm=1.0))
print("sin_clip → tiende a loss=nan; con_clip acota la norma del gradiente")

## 5. Clipping manual en un custom training loop

`tf.clip_by_global_norm` trata todos los gradientes como un único vector.

In [ ]:
modelo = mlp(keras.optimizers.Adam(1e-3))
X = tf.constant(np.random.default_rng(0).normal(size=(128, 784)), dtype=tf.float32)
y = tf.constant(np.random.default_rng(0).integers(0, 10, size=128))
opt = keras.optimizers.Adam(1e-3)

normas = []
for step in range(5):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(y, modelo(X, training=True)))
    grads = tape.gradient(loss, modelo.trainable_variables)
    grads, norma = tf.clip_by_global_norm(grads, 1.0)          # recorte manual
    opt.apply_gradients(zip(grads, modelo.trainable_variables))
    normas.append(float(norma))
print("norma global pre-clip por step:", [round(n, 3) for n in normas])

## Ejercicios

1. **Forzar exploding**: entrená con `Adam(lr=10.0)` y observá `loss=nan`.
2. **Clipping al rescate**: repetí con `Adam(lr=10.0, clipnorm=1.0)`.
3. **`clipnorm` vs `clipvalue`**: compará ambos con un LR razonable.
4. **Monitoreo**: en un custom loop, graficá `||grad||` por step y verificá que queda acotada al `clipnorm` configurado.

## Conclusiones

- **`clipnorm`** reescala el gradiente si supera un umbral, **preservando la dirección**: es el default moderno.
- **`clipvalue`** recorta por elemento y puede cambiar la dirección.
- En custom loops se usa `tf.clip_by_global_norm(grads, c)` (norma global sobre todos los pesos).
- En Transformers/LLMs `clipnorm=1.0` es estándar; en RNN/LSTM clásicos `5.0`.
- El clipping no arregla un LR mal calibrado: si se activa siempre, hay un problema más profundo.

## ✅ Soluciones de los ejercicios

Gradient clipping: forzar exploding, clipping al rescate, `clipnorm` vs `clipvalue`, custom loop con `clip_by_global_norm` y monitoreo de la norma. Sin TF se validan por AST.

**Ej. 1 — Forzar exploding.** `Adam(lr=10.0)` sobre Fashion-MNIST: la loss va a nan rápido.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
model.compile(optimizer=keras.optimizers.Adam(10.0), loss="sparse_categorical_crossentropy")
h = model.fit(Xtr, ytr, epochs=1, verbose=0)
print("loss con Adam(lr=10):", h.history["loss"][0], "-> nan/inf (exploding).")

**Ej. 2 — Clipping al rescate.** `clipnorm=1.0` evita el nan; **ojo**: no arregla un LR mal calibrado, solo la explosión.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
model.compile(optimizer=keras.optimizers.Adam(10.0, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
h = model.fit(Xtr, ytr, epochs=1, verbose=0)
print("loss con clipnorm=1.0:", h.history["loss"][0], "-> finito (no explota).")
print("El clipping evita el nan, pero LR=10 sigue siendo malo: hay que tunear el LR igual.")

**Ej. 3 — `clipnorm` vs `clipvalue`.** Con LR razonable son ~equivalentes; difieren en cómo recortan.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make(**clip):
    m = keras.Sequential([keras.Input((784,)),
        layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(1e-3, **clip),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for clip in [{"clipnorm": 1.0}, {"clipvalue": 0.5}]:
    h = make(**clip).fit(Xtr, ytr, epochs=5, validation_split=0.2, verbose=0)
    print(clip, "-> val_acc =", round(h.history["val_accuracy"][-1], 3))
print("clipnorm reescala el vector entero si su norma pasa el umbral; clipvalue recorta componente a componente.")

**Ej. 4 — Custom loop.** Clipping manual con `tf.clip_by_global_norm(grads, 1.0)` antes de `apply_gradients`.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
opt = keras.optimizers.Adam(1e-3); loss_fn = keras.losses.SparseCategoricalCrossentropy()
ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).batch(128)
gnorm = None
for xb, yb in ds.take(50):
    with tf.GradientTape() as tape:
        loss = loss_fn(yb, model(xb, training=True))
    grads = tape.gradient(loss, model.trainable_variables)
    grads, gnorm = tf.clip_by_global_norm(grads, 1.0)      # clipping global
    opt.apply_gradients(zip(grads, model.trainable_variables))
print("Custom loop con clip_by_global_norm(grads, 1.0). ultima norma global:", float(gnorm))

**Ej. 5 — Monitoreo.** Graficamos la norma del gradiente por step; tras el clip nunca supera el `clipnorm` configurado.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,)),
    layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
opt = keras.optimizers.Adam(1e-3); loss_fn = keras.losses.SparseCategoricalCrossentropy()
ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).batch(128)
norms = []
for xb, yb in ds.take(80):
    with tf.GradientTape() as tape:
        loss = loss_fn(yb, model(xb, training=True))
    grads = tape.gradient(loss, model.trainable_variables)
    _, gnorm = tf.clip_by_global_norm(grads, 1.0)
    norms.append(min(float(gnorm), 1.0))              # norma efectiva post-clip
    opt.apply_gradients(zip(grads, model.trainable_variables))
plt.plot(norms); plt.axhline(1.0, color="r", ls="--", label="clipnorm=1.0")
plt.xlabel("step"); plt.ylabel("||grad|| post-clip"); plt.legend()
plt.title("La norma post-clip nunca supera el umbral"); plt.show()